In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

In [4]:
print("Iniciando a agregação de features analíticas...")

try:
    # Localiza o data warehouse
    current_path = Path(".").resolve()
    db_path = None
    for folder in [current_path] + list(current_path.parents):
        if (folder / "aposta_ganha_dw.db").exists():
            db_path = folder / "aposta_ganha_dw.db"
            break

    if not db_path:
        raise FileNotFoundError("Base de dados não encontrada")
    
    # conectada o banco de dados
    conn = sqlite3.connect(str(db_path))
    cursor = conn.cursor()

    # feature engineering
    cursor.executescript("""
        DROP TABLE IF EXISTS tabela_analytics;
        
        CREATE TABLE tabela_analytics AS
                         
        -- descobre a data mais recente do banco para calcular o recency de forma dinâmica
        WITH data_referencia AS (
            SELECT MAX(data_transacao) as max_date FROM vw_transacoes_limpas
        ),
                         
        -- Agregações clássicas e comportamentais
        metricas_comportamentais AS (
            SELECT
                user_id,
                -- contagem e volumes
                COUNT(CASE WHEN tipo_transacao IN ('aposta_esportiva', 'aposta_cassino') THEN 1 END) AS total_apostas,
                SUM(CASE WHEN tipo_transacao = 'deposito' THEN valor_transacao ELSE 0 END) AS volume_deposito,
                SUM(CASE WHEN tipo_transacao = 'saque' THEN valor_transacao ELSE 0 END) AS volume_saque,
                SUM(valor_ganho) - SUM(valor_perdido) AS saldo_apostas,
                         
                -- recency (dias desde a última aposta)         
                CAST(
                    julianday((SELECT max_date FROM data_referencia)) -
                    julianday(MAX(CASE WHEN tipo_transacao IN ('aposta_esportiva', 'aposta_cassino') THEN data_transacao END))
                    AS INTEGER) AS recency,
                         
                -- deposit velocity (frequência média de depósitos dividida pelos dias ativos)
                CAST(
                    SUM(CASE WHEN tipo_transacao = 'deposito' THEN 1 ELSE 0 END) * 1.0 /
                    MAX(1, CAST(julianday(MAX(data_transacao)) - julianday(MIN(data_transacao)) AS INTEGER))
                AS REAL) AS deposit_velocity
            FROM vw_transacoes_limpas
            GROUP BY user_id
        ),
                         
        -- feature usando o window functions
        chasing_losses_calc AS (
            SELECT
                user_id,
                SUM(is_chasing) AS chasing_losses_events
            FROM (
                SELECT
                    user_id,
                    CASE
                        WHEN tipo_transacao = 'deposito' AND LAG(valor_perdido) OVER (PARTITION BY user_id ORDER BY data_hora) > 0 THEN 1
                        ELSE 0
                    END as is_chasing
                    FROM vw_transacoes_limpas
            )
            GROUP BY user_id
        )

        -- unindo os perfis com as métricas para o machine learning
        SELECT
            u.user_id,
            u.nome,
            u.estado,
            u.preferencia_jogo,
            u.flag_vip,
            COALESCE(mc.total_apostas, 0) AS total_apostas,
            ROUND(COALESCE(mc.volume_deposito, 0), 2) AS volume_deposito,
            ROUND(COALESCE(mc.volume_saque, 0), 2) AS volume_saque,
            ROUND(COALESCE(mc.saldo_apostas, 0), 2) AS saldo_apostas,
            COALESCE(mc.recency, 999) AS recency,
            ROUND(COALESCE(mc.deposit_velocity, 0), 4) AS deposit_velocity,
            COALESCE(cl.chasing_losses_events, 0) AS chasing_losses_events,

            -- criação da variável alvo (target) para o modelo de churn
            -- se o usuário não aposta há mais de 14 dias, classificamos como invalido(1)             
            CASE WHEN COALESCE(mc.recency, 999) > 14 THEN 1 ELSE 0 END AS target_churn
                         
        FROM dim_usuarios u
        LEFT JOIN metricas_comportamentais mc ON u.user_id = mc.user_id
        LEFT JOIN chasing_losses_calc cl ON u.user_id = cl.user_id;                     
    """)

    conn.commit()
    print("Tabela de features 'tabela_analytics' gerada com sucesso")

    # validação dos dados
    print("Pré-visualização das Features")
    df_analytics = pd.read_sql_query("SELECT * FROM tabela_analytics LIMIT 5;", conn)
    print()
    print(df_analytics.to_string(index=False))

    conn.close()

except Exception as e:
    print(f"Falha na agregação de features: {e}")

Iniciando a agregação de features analíticas...
Tabela de features 'tabela_analytics' gerada com sucesso
Pré-visualização das Features

 user_id                  nome estado preferencia_jogo  flag_vip  total_apostas  volume_deposito  volume_saque  saldo_apostas  recency  deposit_velocity  chasing_losses_events  target_churn
  100037       Bárbara Moreira     AM         esportes         0              2             0.00          0.00         101.41        1               0.0                      0             0
  100054   Vitor Hugo Siqueira     PE          cassino         0              1             0.00          0.00          35.49        1               0.0                      0             0
  100063 Srta. Ana Luiza Pires     MG         esportes         0              6            33.45         75.64          22.63        0               1.0                      0             0
  100089       Augusto Moreira     SE         esportes         0              0            36.45        